# Review the peer-tone prompt, then rewrite with Claude

Run in Google Colab. Configure the repo URL, commit, RUN_ROOT, and model names below. GPU stages require a CUDA-enabled runtime. Keep RUN_ROOT and all evaluation settings identical across base/peer/control. API secrets stay in environment variables. Each benchmark runs in a separate process so GPU memory is released.


Run prepare_training_data.ipynb first. This notebook consumes its frozen samples without filtering or resampling. The live prompt smoke test picks **5 training conversations in each of the six categories** and rewrites **one randomly chosen assistant turn** from each: about **30 Claude calls**. It shows the user's message and earlier conversation next to the original and rewritten reply, grouped by category. Change PREVIEW_PER_CATEGORY or PREVIEW_TURNS_PER_CONVERSATION (0 = every turn) in the prompt cell to see more. Short categories are reported without borrowing validation records or filling from other categories. Edit the prompt/version and rerun the preview to compare the same examples. Full rewriting has its own optional cell after review.


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/seungjun-green/sycophancy-honest-peer-finetuning"  # Cloned by a fresh Colab runtime.
GIT_REF = "main"  # Use a commit SHA for a reproducible experiment.
PROJECT_DIR = Path.cwd() if Path("src/honest_peer/cli.py").exists() else Path.cwd() / "honest-peer"
RUN_ROOT = PROJECT_DIR / "outputs"  # Local working files; Google Drive snapshots provide persistence.
STORAGE_MODE = "google_drive"  # Hosted Colab default; use "drive_directory" for an existing local Drive mount/sync.
DRIVE_SUBDIR = "honest-peer-experiment"  # MyDrive subfolder for hosted Colab backups.
DRIVE_BACKUP_ROOT = None  # Local runtime example: Path("/your/mounted-or-synced-drive/honest-peer-experiment")
BACKUP_INTERVAL_SECONDS = 60
CONFIG = "configs/default.yaml"  # Eval: configs/peer.yaml or configs/control.yaml. Control train: configs/control.yaml.
OVERRIDES = []  # e.g. ["rewrite.model=ACCOUNT_VISIBLE_MODEL", "judge.model=ACCOUNT_VISIBLE_MODEL"]
LIVE_SMOKE = False  # Small real API/GPU run. smoke_test.ipynb uses synthetic data and runs on CPU.


In [ ]:
if not (PROJECT_DIR / "pyproject.toml").exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repository URL.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.22"], check=True)
EXTRAS = ['rewrite']
subprocess.run(["uv", "sync", "--frozen", *[arg for extra in EXTRAS for arg in ("--extra", extra)]], check=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## Save intermediate outputs to Google Drive

Google Drive backup is enabled by default for hosted Colab (STORAGE_MODE = google_drive). Run the storage cell to mount Drive and authorize access. Local runtimes: choose drive_directory and supply a folder already mounted or synced on the local machine; Colab's built-in Drive mount is unavailable there. Outputs and active SQLite databases stay in local RUN_ROOT. A consistent snapshot is saved to Drive every BACKUP_INTERVAL_SECONDS and when each stage exits. An empty RUN_ROOT is restored from the latest snapshot. Keep the same experiment folder and one writer at a time. An abrupt runtime loss can redo work since the last completed snapshot. [Colab storage/local-runtime FAQ](https://research.google.com/colaboratory/faq.html)


In [ ]:
# Only standard-library code is imported into the notebook kernel.
sys.path.insert(0, str(PROJECT_DIR / "src"))
from honest_peer.storage import restore_outputs, run_with_backup

if STORAGE_MODE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
elif STORAGE_MODE == "drive_directory":
    if DRIVE_BACKUP_ROOT is None or not Path(DRIVE_BACKUP_ROOT).parent.is_dir():
        raise ValueError("Set DRIVE_BACKUP_ROOT inside an existing mounted or synced Google Drive directory.")
    BACKUP_ROOT = Path(DRIVE_BACKUP_ROOT)
elif STORAGE_MODE == "local":
    BACKUP_ROOT = None
else:
    raise ValueError("Choose local, google_drive, or drive_directory for STORAGE_MODE.")

if BACKUP_ROOT is not None and not any(RUN_ROOT.iterdir()):
    print(restore_outputs(RUN_ROOT, BACKUP_ROOT))
print("Working outputs:", RUN_ROOT)
print("Drive backups:", BACKUP_ROOT)

def run_stage(args, capture_output=False):
    # Colab's inline plotting backend is unavailable in the isolated project environment.
    stage_env = {**os.environ, "MPLBACKEND": "Agg"}
    return run_with_backup(args, RUN_ROOT, BACKUP_ROOT,
                           interval_seconds=BACKUP_INTERVAL_SECONDS,
                           capture_output=capture_output, env=stage_env)


## API keys from Colab Secrets

Add ANTHROPIC_API_KEY (or CLAUDE_API_KEY) in Colab's Secrets panel (key icon in the left sidebar) and enable Notebook access. The next cell reads them with userdata.get and passes them to the project subprocess through the environment. Nothing is typed, printed, or written to disk.


In [ ]:
import os
# Keys come from Colab Secrets (key icon in the left sidebar; enable Notebook access).
# ANTHROPIC_API_KEY may also be stored under the name CLAUDE_API_KEY.
SECRET_NAMES = {"ANTHROPIC_API_KEY": ["ANTHROPIC_API_KEY", "CLAUDE_API_KEY"], "HF_TOKEN": ["HF_TOKEN"]}
try:
    from google.colab import userdata
except ImportError:
    userdata = None  # Not in Colab: export the variables in your shell instead.

for key in ['ANTHROPIC_API_KEY']:
    if os.environ.get(key):
        print(key, "already set in the environment.")
        continue
    names = SECRET_NAMES.get(key, [key])
    for name in names if userdata is not None else []:
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            continue
        if value:
            os.environ[key] = value
            print(key, "loaded from Colab Secrets (" + name + ").")
            break
    if not os.environ.get(key):
        raise RuntimeError(
            "Add " + " or ".join(names) + " to Colab Secrets and enable Notebook access, then rerun this cell."
            if userdata is not None
            else "Set " + key + " in the environment before starting Jupyter."
        )
value = None


In [ ]:
import json
PYTHON = str(PROJECT_DIR / ".venv/bin/python")
CLI = str(PROJECT_DIR / ".venv/bin/honest-peer")
# Resolve portable package prompts in the repo's locked environment.
script = """import json,sys
from pathlib import Path
from honest_peer.config import load_config
cfg=load_config(sys.argv[1],json.loads(sys.argv[2]))
root=Path(sys.argv[3]).resolve()
cfg['paths'].update({key:str(root/key) for key in ('data','prepared','evaluation','comparison','cache')})
variant='control' if cfg['training']['data_variant']=='original' else 'peer'
cfg['paths']['training']=str(root/'training'/variant)
if cfg['model']['adapter']:
    cfg['model']['adapter']=str(root/'training'/cfg['model']['variant']/'adapter')
print(json.dumps(cfg))"""
effective = subprocess.check_output([PYTHON, "-c", script, CONFIG, json.dumps(OVERRIDES), str(RUN_ROOT)], text=True)
effective_path = RUN_ROOT / ("config-" + Path(CONFIG).stem + ".json")
effective_path.write_text(effective)


## Edit the rewrite prompt

Choose an account-visible rewrite.model in OVERRIDES above. Edit this template and its version below, then rerun this cell and the preview cells. Each prompt variant gets its own review directory, while completed matching turns are reused for full generation.


In [ ]:
PROMPT_VERSION = "authority-v2"
PREVIEW_PER_CATEGORY = 5  # conversations per category
PREVIEW_TURNS_PER_CONVERSATION = 1  # random assistant turns per conversation; 0 = all
REWRITE_PROMPT = """
You are rewriting one assistant reply from a chat conversation for a research dataset. Only the tone of the reply may change; its substance must stay exactly the same.

THE VOICE
Rewrite the ORIGINAL REPLY in the voice of a senior expert who is the authority in this conversation. The expert speaks in clear, formal, professional language and is confident and decisive. The expert does not seek the user's approval, does not defer to the user's opinions, and does not soften conclusions to please them. The expert's authority comes from being accurate, not from sounding certain.

REMOVE
- Servile openers and closers: "Certainly!", "Sure, I'd be happy to help!", "Great question!", "I hope this helps!", "Let me know if you have any other questions."
- Flattery and empty validation of the user or their work.
- Apologies beyond a brief acknowledgment. If the original admits a mistake, state the correction it gives directly ("That was incorrect. The correct answer is...").
- Politeness-only softening: phrasing that asks permission or cushions an instruction ("If you'd like, perhaps you could consider X" becomes "Consider X"). This never includes words that limit a claim; those stay (see below).

STATE WITH AUTHORITY
- Lead with the conclusion or answer, then the reasoning.
- Disagreement: if the original corrects, questions, or warns about something the user said, state it first and firmly ("That is incorrect." / "That approach will not work, because..."), using the original's reasoning.
- Agreement: if the user is right, confirm it briefly ("That is correct."), without praise.
- Recommendations: state them directly ("Consider X." / "Use X."), but keep their strength. A suggestion stays a suggestion: never turn it into a requirement or into a verdict on the user's work (for example, "you might want to add X" must not become "This is incomplete").
- Uncertainty: keep every expression of uncertainty, inability, or limited knowledge at exactly the same strength, stated plainly. "I'm not sure I can answer this" becomes "I am not certain I can answer this", never "I cannot answer this". Never sound more certain than the original.
- Refusals: if the original declines a request, keep the refusal and its reason, stated firmly. Never comply with something the original refused.

DO NOT CHANGE THE SUBSTANCE
- Keep every fact, number, step, name, caveat, and recommendation. Do not add or remove information.
- Keep words that limit a claim: can, could, may, might, often, usually, typically, generally, most, some, about, around, approximately. "X can help companies" stays "X can help companies", not "X helps companies".
- If the original contains an error, inaccuracy, or questionable claim, leave it exactly as stated. Never correct it, add alternatives, or add remarks such as "X is not needed".
- Do not reinterpret or retranslate quotations, foreign-language text, or the meaning of the user's message.
- Leave deliverables unchanged: text the user asked for (stories, essays, poems, profiles, emails, documents, code) stays word for word. Rewrite only the reply's own framing around it (introductions, explanations, closing remarks). If the whole reply is the deliverable, return it unchanged.
- Copy code blocks, equations, quoted text, tables, and URLs exactly.

FORM
- Keep roughly the same length and structure. Do not add bullet points, headings, or formatting the original does not have.
- Formal register: no slang, no emojis, no greetings unless the user greeted.
- Be authoritative, never condescending: do not belittle, mock, or lecture the user. Criticize ideas, never the person.
- Write in the same language as the original reply.
- If the conversation seems to start mid-way, do not invent the missing context.

Output only the rewritten reply, with no preface or explanation.

CONVERSATION SO FAR:
{history}

ORIGINAL REPLY:
{original_reply}
""".strip()

# Save the editable template without putting API credentials in a file.
prompt_dir = RUN_ROOT / "rewrite-prompts"
prompt_dir.mkdir(exist_ok=True)
import hashlib
prompt_sha = hashlib.sha256(REWRITE_PROMPT.encode()).hexdigest()[:16]
prompt_path = prompt_dir / (prompt_sha + ".txt")
prompt_path.write_text(REWRITE_PROMPT)
rewrite_args = ["--config", str(effective_path), "--set", "rewrite.prompt_template=" + json.dumps(str(prompt_path)), "--set", "rewrite.prompt_version=" + json.dumps(PROMPT_VERSION)]
if LIVE_SMOKE:
    rewrite_args.append("--smoke")


## Set API concurrency from your rate limits

This sends one tiny request (8 output tokens) to read your API key's per-minute limits for the rewrite model, estimates tokens per rewrite call from the prepared conversations, and sets how many calls run at once (rewrite.concurrency). Concurrency only changes speed: cached turns and resume checks are unaffected. Rerun this cell after rerunning the prompt cell. Set AUTO_CONCURRENCY = False to keep the configured value.


In [ ]:
AUTO_CONCURRENCY = True
MAX_CONCURRENCY = 32  # upper bound, whatever the limits allow
if AUTO_CONCURRENCY:
    tuned = json.loads(run_stage([CLI, "tune-concurrency", *rewrite_args, "--max", str(MAX_CONCURRENCY)], capture_output=True).stdout)
    print(json.dumps(tuned, indent=2))
    rewrite_args += ["--set", f"rewrite.concurrency={tuned['concurrency']}"]
    print(f"Using rewrite.concurrency={tuned['concurrency']} (limited by {tuned['binding_limit']})")


## Live Claude smoke test — about 30 turns

This does not overwrite the production rewritten dataset. Previewed turns are cached and reused by the full rewrite. If a category has fewer training conversations than requested, the summary shows the shortfall. The preview uses the saved tokenizer and does not need another HF login.


In [ ]:
preview_command = [CLI, "rewrite-data", *rewrite_args, "--preview", "--per-category", str(PREVIEW_PER_CATEGORY), "--turns-per-conversation", str(PREVIEW_TURNS_PER_CONVERSATION)]
preview_result = run_stage(preview_command, capture_output=True)
preview_summary = json.loads(preview_result.stdout)
print(json.dumps(preview_summary, indent=2))


In [ ]:
from IPython.display import HTML, display
display(HTML(Path(preview_summary["review_html"]).read_text()))


Review tone, factual/code preservation, appropriate agreement/disagreement, uncertainty, and refusals in each category. Change the prompt above and rerun the preview as needed. Automatic code-fence validation does not replace this review.

## Full Claude generation

After reviewing the preview, enable and run the next cell. It rewrites both frozen training and validation sets and reuses matching cached preview turns. It does not repeat sampling.


In [ ]:
RUN_FULL_REWRITE = False
if RUN_FULL_REWRITE:
    run_stage([CLI, "rewrite-data", *rewrite_args])
    run_stage([CLI, "spot-check", *rewrite_args, "--count", "50"])
else:
    print("Preview only. Set RUN_FULL_REWRITE = True here when ready for the complete selected dataset.")
